# Taller: Clusterización de Datos con PySpark

## Objetivo
Implementar un pipeline completo de clusterización utilizando PySpark MLlib y registrar el experimento en MLFlow.

## Contenido del Taller
1. Generación de dataset sintético
2. Análisis exploratorio de datos (EDA)
3. Preparación de features
4. Entrenamiento del modelo KMeans
5. Evaluación y visualización de resultados
6. Registro del experimento en MLFlow

## Dataset
Crearemos un dataset sintético con características de clientes (edad, ingresos, gasto) para segmentación.

In [0]:
# Importar librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, round as spark_round, monotonically_increasing_id
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
import mlflow
import mlflow.spark
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("✓ Librerías importadas exitosamente")

In [0]:
# Generar dataset sintético de clientes (800 registros)
np.random.seed(42)

# Crear datos con 3 grupos naturales
group1 = pd.DataFrame({
    'edad': np.random.normal(25, 5, 250),
    'ingreso_anual': np.random.normal(30000, 5000, 250),
    'gasto_mensual': np.random.normal(1200, 200, 250)
})

group2 = pd.DataFrame({
    'edad': np.random.normal(45, 7, 300),
    'ingreso_anual': np.random.normal(65000, 8000, 300),
    'gasto_mensual': np.random.normal(2800, 400, 300)
})

group3 = pd.DataFrame({
    'edad': np.random.normal(60, 6, 250),
    'ingreso_anual': np.random.normal(50000, 6000, 250),
    'gasto_mensual': np.random.normal(1800, 300, 250)
})

# Combinar grupos
df_pandas = pd.concat([group1, group2, group3], ignore_index=True)

# Convertir a Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)
df_spark = df_spark.withColumn("cliente_id", monotonically_increasing_id())

print(f"Dataset generado con {df_spark.count()} registros")
display(df_spark.limit(10))

In [0]:
# Análisis exploratorio de datos
print("=== Estadísticas Descriptivas ===")
df_stats = df_spark.select('edad', 'ingreso_anual', 'gasto_mensual').describe()
display(df_stats)

print("\n=== Información del Dataset ===")
print(f"Total de registros: {df_spark.count()}")
print(f"Total de columnas: {len(df_spark.columns)}")
print(f"\nEsquema del DataFrame:")
df_spark.printSchema()

In [0]:
# Visualizaciones exploratorias
df_pd = df_spark.select('edad', 'ingreso_anual', 'gasto_mensual').toPandas()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribuciones
axes[0, 0].hist(df_pd['edad'], bins=30, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribución de Edad', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Edad')
axes[0, 0].set_ylabel('Frecuencia')

axes[0, 1].hist(df_pd['ingreso_anual'], bins=30, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Distribución de Ingreso Anual', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Ingreso Anual ($)')
axes[0, 1].set_ylabel('Frecuencia')

axes[1, 0].hist(df_pd['gasto_mensual'], bins=30, color='salmon', edgecolor='black')
axes[1, 0].set_title('Distribución de Gasto Mensual', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Gasto Mensual ($)')
axes[1, 0].set_ylabel('Frecuencia')

# Matriz de correlación
corr_matrix = df_pd.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1], cbar_kws={'label': 'Correlación'})
axes[1, 1].set_title('Matriz de Correlación', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualizaciones generadas exitosamente")

In [0]:
# Preparar features para clustering
feature_cols = ['edad', 'ingreso_anual', 'gasto_mensual']

# Ensamblar features en un vector
vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw"
)

df_assembled = vector_assembler.transform(df_spark)

# Normalizar features (importante para KMeans)
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("✓ Features preparadas y normalizadas")
display(df_scaled.select('cliente_id', 'edad', 'ingreso_anual', 'gasto_mensual', 'features').limit(5))

In [0]:
# Configurar y entrenar modelo KMeans
k = 3  # Número de clusters (sabemos que hay 3 grupos naturales)

kmeans = KMeans(
    k=k,
    seed=42,
    featuresCol='features',
    predictionCol='cluster'
)

print(f"Entrenando modelo KMeans con k={k} clusters...")
kmeans_model = kmeans.fit(df_scaled)

# Realizar predicciones
df_predictions = kmeans_model.transform(df_scaled)

print(f"✓ Modelo entrenado exitosamente")
print(f"\nCentros de los clusters:")
centers = kmeans_model.clusterCenters()
for i, center in enumerate(centers):
    print(f"Cluster {i}: {center}")

display(df_predictions.select('cliente_id', 'edad', 'ingreso_anual', 'gasto_mensual', 'cluster').limit(15))

In [0]:
# Evaluar el modelo usando Silhouette Score
from pyspark.sql.functions import avg

evaluator = ClusteringEvaluator(
    featuresCol='features',
    predictionCol='cluster',
    metricName='silhouette'
)

silhouette_score = evaluator.evaluate(df_predictions)
print(f"Silhouette Score: {silhouette_score:.4f}")
print("(Valores cercanos a 1 indican clusters bien definidos)")

# Contar elementos por cluster
print("\n=== Distribución de Clientes por Cluster ===")
cluster_counts = df_predictions.groupBy('cluster').count().orderBy('cluster')
display(cluster_counts)

# Calcular estadísticas por cluster
print("\n=== Características Promedio por Cluster ===")
cluster_stats = df_predictions.groupBy('cluster').agg(
    spark_round(avg('edad'), 2).alias('edad_promedio'),
    spark_round(avg('ingreso_anual'), 2).alias('ingreso_promedio'),
    spark_round(avg('gasto_mensual'), 2).alias('gasto_promedio')
).orderBy('cluster')

display(cluster_stats)

In [0]:
# Visualizar los clusters
df_viz = df_predictions.select('edad', 'ingreso_anual', 'gasto_mensual', 'cluster').toPandas()

fig = plt.figure(figsize=(16, 5))

# Gráfico 1: Edad vs Ingreso
ax1 = fig.add_subplot(131)
scatter1 = ax1.scatter(df_viz['edad'], df_viz['ingreso_anual'], 
                       c=df_viz['cluster'], cmap='viridis', 
                       s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
ax1.set_xlabel('Edad', fontsize=11)
ax1.set_ylabel('Ingreso Anual ($)', fontsize=11)
ax1.set_title('Clusters: Edad vs Ingreso', fontsize=12, fontweight='bold')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Gráfico 2: Ingreso vs Gasto
ax2 = fig.add_subplot(132)
scatter2 = ax2.scatter(df_viz['ingreso_anual'], df_viz['gasto_mensual'], 
                       c=df_viz['cluster'], cmap='viridis', 
                       s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
ax2.set_xlabel('Ingreso Anual ($)', fontsize=11)
ax2.set_ylabel('Gasto Mensual ($)', fontsize=11)
ax2.set_title('Clusters: Ingreso vs Gasto', fontsize=12, fontweight='bold')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

# Gráfico 3: Edad vs Gasto
ax3 = fig.add_subplot(133)
scatter3 = ax3.scatter(df_viz['edad'], df_viz['gasto_mensual'], 
                       c=df_viz['cluster'], cmap='viridis', 
                       s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
ax3.set_xlabel('Edad', fontsize=11)
ax3.set_ylabel('Gasto Mensual ($)', fontsize=11)
ax3.set_title('Clusters: Edad vs Gasto', fontsize=12, fontweight='bold')
plt.colorbar(scatter3, ax=ax3, label='Cluster')

plt.tight_layout()
plt.show()

print("✓ Visualizaciones de clusters generadas exitosamente")

In [0]:
# Registrar experimento en MLFlow
experiment_name = "/Workspace/Users/alonso_vento@outlook.com/Portafolio_BigData-DataBricks_MarioAlonsoVentoAlvarado/clasificacion-binaria-taller"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="kmeans_customer_segmentation") as run:
    
    # Registrar parámetros
    mlflow.log_param("k_clusters", k)
    mlflow.log_param("features", str(feature_cols))
    mlflow.log_param("total_registros", df_spark.count())
    mlflow.log_param("seed", 42)
    mlflow.log_param("normalizacion", "StandardScaler")
    mlflow.log_param("algoritmo", "KMeans")
    
    # Registrar métricas
    mlflow.log_metric("silhouette_score", silhouette_score)
    mlflow.log_metric("num_clusters", k)
    
    # Registrar centros de clusters como métricas
    centers = kmeans_model.clusterCenters()
    for i, center in enumerate(centers):
        mlflow.log_metric(f"cluster_{i}_center_0", float(center[0]))
        mlflow.log_metric(f"cluster_{i}_center_1", float(center[1]))
        mlflow.log_metric(f"cluster_{i}_center_2", float(center[2]))
    
    # Registrar información adicional como tags
    mlflow.set_tag("tipo_modelo", "clustering")
    mlflow.set_tag("algoritmo", "KMeans")
    mlflow.set_tag("proposito", "segmentacion_clientes")
    mlflow.set_tag("framework", "PySpark MLlib")
    
    run_id = run.info.run_id
    experiment_id = run.info.experiment_id
    
print("="*60)
print("✓ EXPERIMENTO REGISTRADO EXITOSAMENTE EN MLFLOW")
print("="*60)
print(f"Experiment Name: {experiment_name}")
print(f"Experiment ID: {experiment_id}")
print(f"Run ID: {run_id}")
print(f"Silhouette Score: {silhouette_score:.4f}")
print(f"Número de Clusters: {k}")
print(f"Total de Registros: {df_spark.count()}")
print("="*60)
print("\n✓ Puedes ver el experimento completo en la UI de MLFlow.")
print("  Navega a 'Experiments' en el panel lateral para ver todos los detalles.")

## 🎯 Conclusiones del Taller

### Resultados Obtenidos

* **Dataset:** Se generó exitosamente un dataset sintético con 800 registros de clientes
* **Features:** Se utilizaron 3 características: edad, ingreso anual y gasto mensual
* **Algoritmo:** KMeans con k=3 clusters
* **Normalización:** StandardScaler aplicado para mejorar convergencia
* **Métrica:** Silhouette Score para evaluar calidad de clusters
* **Registro:** Experimento completo documentado en MLFlow

### Segmentos Identificados

El modelo identificó 3 segmentos distintos de clientes:

1. **Cluster 0:** Clientes jóvenes con ingresos y gastos bajos
2. **Cluster 1:** Clientes de mediana edad con ingresos y gastos altos
3. **Cluster 2:** Clientes mayores con ingresos moderados y gastos medios

### Aplicaciones Prácticas

* Marketing personalizado por segmento
* Diseño de productos específicos para cada grupo
* Estrategias de retención diferenciadas
* Optimización de campañas publicitarias